In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window, DataFrame
import torch
import pandas as pd
import tiktoken

torch.manual_seed(123)

src_path = Path.cwd().parent / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to sys.path: {src_path}")

load_dotenv()  # reads .env file from the current directory
spark = SparkSession.builder.getOrCreate()

Added to sys.path: /home/jtv/code/jtviegas/languagemodels/src


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/04 22:53:50 WARN Utils: Your hostname, jtv, resolves to a loopback address: 127.0.1.1; using 192.168.0.160 instead (on interface wlp192s0)
26/06/04 22:53:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/04 22:53:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# constants

In [2]:
DEFAULT_EMBEDDINGS_DIMENSION = 256
DEFAULT_SEQUENCE_LENGTH = 4
DEFAULT_STRIDE = 1

path_data = Path.cwd().parent / "data"
path_narratives = path_data / "faers" / "faers_narratives_small"
path_gpt2_settings = path_data / "model_weights" / "gpt2" / "124M" / "settings.pickle.gz"
path_gpt2_parameters = path_data / "model_weights" / "gpt2" / "124M" / "parameters.pickle.gz"

# data

In [3]:
def get_faers_narratives(subset_length: int = 1000) -> pd.DataFrame:
    return spark.read.load(str(path_narratives)).limit(subset_length).select("text").toPandas()

In [4]:
def get_dummy_corpus() -> pd.DataFrame:
    return pd.DataFrame({
        "text": [
            "The cat is on the table.",
            "The dog is in the garden.",
            "The bird is flying in the sky.",
            "The fish is swimming in the pond."
        ]
    })

# input preparation

In [5]:
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.eot_token)
print(tokenizer.n_vocab)

50256
50257


In [6]:
from tgedr_languagemodels.utils_llm import harmonize_text_sequences

harmonize_text_sequences(get_dummy_corpus(), tokenizer)


[[464, 3797, 318, 319, 262, 3084, 13, 50256],
 [464, 3290, 318, 287, 262, 11376, 13, 50256],
 [464, 6512, 318, 7348, 287, 262, 6766, 13],
 [464, 5916, 318, 14899, 287, 262, 16723, 13]]

In [7]:
model_config = {
    "vocabulary_size": tokenizer.n_vocab,
    "embeddings_dimension": DEFAULT_EMBEDDINGS_DIMENSION,
    "sequence_length": DEFAULT_SEQUENCE_LENGTH,
    "stride": DEFAULT_STRIDE
}

In [17]:
from tgedr_languagemodels.utils_llm import save_pickle_compressed, load_pickle_compressed

params = load_pickle_compressed(model_weights_parameters)
settings = load_pickle_compressed(model_weights_settings)

# model

In [18]:
from tgedr_languagemodels.models import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"

NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024})
NEW_CONFIG.update({"qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feat

In [19]:
from tgedr_languagemodels.model_weights import load_weights_into_gpt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, params)
gpt.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feat

In [20]:
from tgedr_languagemodels.utils_llm import generate, text_to_token_ids, token_ids_to_text
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")


token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5,
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you along so quickly, because in the old world these moments had little weight. At the end or mid-point, I will


# fine-tuning for classification

## getting the data

In [21]:
path_spam = path_data / "sms_spam_collection"
zipfile_url = str(path_data / "sms_spam_collection.zip")
spam_tsv_url = str(path_spam / "SMSSpamCollection.tsv")
spam_url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

In [22]:
import urllib.request
import zipfile
import os
from pathlib import Path





def download_and_unzip_spam_data(url, zip_path, path_spam, path_data):
    if Path(spam_tsv_url).exists():
        print(f"{spam_tsv_url} already exists. Skipping download "
              "and extraction."
        )
        return

    with urllib.request.urlopen(url) as response:    #1
        with open(zipfile_url, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zipfile_url, "r") as zip_ref:    #2
        zip_ref.extractall(path_spam)

    original_file_path = path_spam / "SMSSpamCollection"
    os.rename(original_file_path, spam_tsv_url)               #3
    print(f"File downloaded and saved as {spam_tsv_url}")

download_and_unzip_spam_data(spam_url, zipfile_url, path_spam, path_data)
#1 Downloads the file
#2 Unzips the file
#3 Adds a .tsv file extension

/home/jtv/code/jtviegas/languagemodels/data/sms_spam_collection/SMSSpamCollection.tsv already exists. Skipping download and extraction.


In [24]:
import pandas as pd
df = pd.read_csv(
    spam_tsv_url, sep="\t", header=None, names=["Label", "Text"]
)
df      #1

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [25]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [26]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]     #1
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam, random_state=123
    )                                         #2
    balanced_df = pd.concat([
        ham_subset, df[df["Label"] == "spam"]
    ])                               #3
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())
#1 Counts the instances of “spam”
#2 Randomly samples “ham” instances to match the number of “spam” instances
#3 Combines ham subset with “spam”

Label
ham     747
spam    747
Name: count, dtype: int64


In [27]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [28]:
def random_split(df, train_frac, validation_frac):

    df = df.sample(
        frac=1, random_state=123
    ).reset_index(drop=True)               #1
    train_end = int(len(df) * train_frac)          #2
    validation_end = train_end + int(len(df) * validation_frac)

 #3
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(
    balanced_df, 0.7, 0.1)                     #4
#1 Shuffles the entire DataFrame
#2 Calculates split indices
#3 Splits the DataFrame
#4 Test size is implied to be 0.2 as the remainder.

In [29]:
train_df.to_csv(str(path_spam / "train.csv"), index=None)
validation_df.to_csv(str(path_spam / "validation.csv"), index=None)
test_df.to_csv(str(path_spam / "test.csv"), index=None)

In [30]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [31]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, pd_df, tokenizer, max_length=None,
                 pad_token_id=50256):
        # self.data = pd.read_csv(csv_file)
        self.data = pd_df

 #1
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
 #2
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

 #3
        self.encoded_texts = [
            encoded_text + [pad_token_id] * 
            (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]


    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
#1 Pretokenizes texts
#2 Truncates sequences if they are longer than max_length
#3 Pads sequences to the longest sequence

In [32]:
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [33]:
train_dataset = SpamDataset(
    pd_df=train_df,
    max_length=None,
    tokenizer=tokenizer
)

In [34]:
print(train_dataset.max_length)

120


In [35]:
val_dataset = SpamDataset(
    pd_df=validation_df,
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    pd_df=test_df,
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

In [36]:
from torch.utils.data import DataLoader

num_workers = 0      #1 This setting ensures compatibility with most computers.
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [37]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [38]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


#### Initializing a model with pretrained weights

In [39]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size": 50257,          #1
    "context_length": 1024,       #2
    "drop_rate": 0.0,             #3
    "qkv_bias": True              #4
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
#1 Vocabulary size
#2 Context length
#3 Dropout rate
#4 Query-key-value bias

In [45]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


def generate_text_simple(
    model,
    idx,  # 1
    max_new_tokens,
    context_size,
    temperature=1.0,
):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]  # 2
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]  # 3
        # probas = torch.softmax(logits, dim=-1)  # 4
        probas = softmax_with_temperature(logits, temperature)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # 5
        idx = torch.cat((idx, idx_next), dim=1)  # 6

    return idx


In [44]:
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feat

In [46]:
text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you!!!!!!!!!!!!!!!
